 # Long audio

In [ ]:
import torch
! pip install datasets
from transformers import AutoProcessor, WhisperForConditionalGeneration
from datasets import load_dataset, Audio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.7/472.7 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 20.2 MB/s eta 0:00:00


In [ ]:
processor = AutoProcessor.from_pretrained("openai/whisper-medium")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:99: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

In [ ]:

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-medium")
model.cuda()

config.json:   0%|          | 0.00/1.99k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/3.75k [00:00<?, ?B/s]

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 1024, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(1024, 1024, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 1024)
      (layers): ModuleList(
        (0-23): 24 x WhisperEncoderLayer(
          (self_attn): WhisperSdpaAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias

In [ ]:
from pathlib import Path
!pip install jiwer
import jiwer
import librosa
import IPython.display as idp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 59.4 MB/s eta 0:00:00


In [ ]:
path_to_content = Path().cwd().parent / "content"
path_to_audio = path_to_content / "lecture1.wav"
test_audio, sr = librosa.load(path_to_audio, sr=16_000)
# sr - sampling rate - частота дискретезации
idp.Audio(test_audio, rate=sr)

In [ ]:

inputs = processor(test_audio, return_tensors="pt", truncation=False, padding="longest", return_attention_mask=True, sampling_rate=16_000)

inputs = inputs.to("cuda", torch.float32)


In [ ]:

# transcribe audio to ids

generated_ids = model.generate(return_timestamps=True, **inputs)

transcription = processor.batch_decode(generated_ids, skip_special_tokens=True)

transcription[0]

Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


' Прошлый раз мы с вами начали рассматривать А теперь надо идти дальше, как говорил покойный Горбачёв, начать сформировать и углубить. Вот давайте углубим то, что у нас было. Значит, мы с вами выяснили, вернее не выяснили, а констатировали, что возраст Вселенной примерно 4 миллиарда, 14 миллиардов лет, 13-й и 70-й. Ну так считается. Мы с вами выяснили, что она образовалась в результате так называемого Большого взрыва. Big Bang. Вот его англичане называют, ну и в литературе вы можете увидеть. Вот этот Big Bang, большой взрыв. Но что взорвалось, не очень ясно. Было какое-то вещество, которое находилось в очень горячем и в очень плотном состоянии, которое нельзя описать просто. И вот это вдруг взорвалось, и мы знаем, ну мы считаем, что мы знаем, примерно после одной секунды в минус сорок третьей степени, так называемое планковское время. И вот с тех пор Вселенная расширяется. Это доказано. Доказано астрономами и физиками, что галактики, которые есть, они удаляются. И чем дальше галактика,

In [ ]:
path_to_doc =  "../content/recognized_1.txt"
with open(path_to_doc, "w", encoding="utf-8") as f:
    f.write(transcription[0])